In [1]:
!pip -q install imbalanced-learn xgboost lightgbm catboost

In [2]:
# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import os
import platform
import subprocess
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from scipy.stats import t, ttest_rel

warnings.filterwarnings("ignore")

# ============================================================
# STEP 3: LOAD THE TWO EXISTING DATASETS
# ============================================================

folder = "/content/drive/MyDrive/BC SURVIVAL METABRIC DATASETS"

clean_file = os.path.join(folder, "cleaned_full_dataset.csv")
original_file = os.path.join(folder, "Breast Cancer METABRIC.csv")

cleaned = pd.read_csv(clean_file)
original = pd.read_csv(original_file)

print("Cleaned dataset shape:", cleaned.shape)
print("Original dataset shape:", original.shape)

# ============================================================
# STEP 4: RECOVER THE SURVIVAL TARGET
# The cleaned dataset's Unnamed: 0 column contains the
# original row positions, which were previously verified.
# ============================================================

if "Unnamed: 0" not in cleaned.columns:
    raise ValueError("The expected 'Unnamed: 0' column is missing.")

if "Overall Survival Status" not in original.columns:
    raise ValueError(
        "The original dataset does not contain "
        "'Overall Survival Status'. Check the column name."
    )

original_positions = cleaned["Unnamed: 0"].astype(int)

if original_positions.duplicated().any():
    raise ValueError("Duplicate original row positions were found.")

if original_positions.min() < 0 or original_positions.max() >= len(original):
    raise ValueError("Some row positions do not match the original dataset.")

survival_status = (
    original.iloc[original_positions]["Overall Survival Status"]
    .reset_index(drop=True)
)

# Encode Living = 0 and Deceased = 1
y = survival_status.map({
    "Living": 0,
    "Deceased": 1
})

if y.isna().any():
    raise ValueError(
        "Some survival labels could not be mapped. "
        "Check the original survival-status values."
    )

y = y.astype(int)

# ============================================================
# STEP 5: PREPARE PREDICTORS
# ============================================================

X = cleaned.drop(columns=["Unnamed: 0"]).copy()

# Remove a target column if one is already present in the
# cleaned file, so it cannot accidentally enter the predictors.
possible_target_columns = [
    col for col in X.columns
    if col.strip().lower() in [
        "overall_survival_status",
        "overall survival status"
    ]
]

if possible_target_columns:
    X = X.drop(columns=possible_target_columns)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors="coerce")

if X.isnull().sum().sum() > 0:
    raise ValueError(
        "Missing or non-numeric predictor values were found. "
        "Stop here and inspect the dataset before modelling."
    )

if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Infinite values were found in the predictors.")

if len(X) != len(y):
    raise ValueError("Predictor and target row counts do not match.")

print("\nPrepared dataset")
print("----------------")
print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("\nSurvival class distribution:")
print(y.value_counts().rename(index={0: "Living", 1: "Deceased"}))

print("\nData preparation completed successfully.")

Mounted at /content/drive
Cleaned dataset shape: (1979, 64)
Original dataset shape: (2509, 34)

Prepared dataset
----------------
Predictor shape: (1979, 63)
Target shape: (1979,)

Survival class distribution:
Overall Survival Status
Deceased    1142
Living       837
Name: count, dtype: int64

Data preparation completed successfully.


In [3]:
# ============================================================
# REPEATED STRATIFIED 5-FOLD CROSS-VALIDATION
# 5 folds x 2 repetitions = 10 evaluation folds per model
# ============================================================

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, roc_auc_score, confusion_matrix
)
from scipy.stats import t as student_t
import time

# ------------------------------------------------------------
# 1. DEFINE THE ORIGINAL MODEL CONFIGURATIONS
# ------------------------------------------------------------

def create_models():

    xgb = XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )

    lgbm = LGBMClassifier(
        random_state=42
    )

    cat = CatBoostClassifier(
        verbose=0,
        random_state=42
    )

    models = {
        "Logistic Regression": LogisticRegression(),

        "Random Forest": RandomForestClassifier(
            random_state=42
        ),

        "XGBoost": XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),

        "LightGBM": LGBMClassifier(
            random_state=42
        ),

        "CatBoost": CatBoostClassifier(
            verbose=0,
            random_state=42
        ),

        "Voting Ensemble": VotingClassifier(
            estimators=[
                ("xgb", xgb),
                ("lgbm", lgbm),
                ("cat", cat)
            ],
            voting="soft"
        )
    }

    return models


# ------------------------------------------------------------
# 2. SET UP THE CROSS-VALIDATION
# ------------------------------------------------------------

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=42
)

results = []

total_folds = 10
start_time = time.time()

# ------------------------------------------------------------
# 3. TRAIN AND EVALUATE EACH MODEL
# ------------------------------------------------------------

for model_name in create_models().keys():

    print("\n" + "=" * 65)
    print("MODEL:", model_name)
    print("=" * 65)

    fold_number = 0

    for train_index, validation_index in cv.split(X, y):

        fold_number += 1

        X_train = X.iloc[train_index]
        X_validation = X.iloc[validation_index]

        y_train = y.iloc[train_index]
        y_validation = y.iloc[validation_index]

        # Re-create the model for each fold
        model = create_models()[model_name]

        # Scaling and SMOTE are fitted/applied only on training data
        pipeline = Pipeline(steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=42)),
            ("classifier", model)
        ])

        pipeline.fit(X_train, y_train)

        # Predict on the untouched validation fold
        y_pred = pipeline.predict(X_validation)

        # Probability of class 1 = Deceased
        y_probability = pipeline.predict_proba(X_validation)[:, 1]

        # Confusion matrix: class 0 = Living, class 1 = Deceased
        tn, fp, fn, tp = confusion_matrix(
            y_validation,
            y_pred,
            labels=[0, 1]
        ).ravel()

        accuracy = accuracy_score(y_validation, y_pred)

        sensitivity = recall_score(
            y_validation, y_pred, pos_label=1, zero_division=0
        )

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        precision = precision_score(
            y_validation, y_pred, pos_label=1, zero_division=0
        )

        f1 = f1_score(
            y_validation, y_pred, pos_label=1, zero_division=0
        )

        auc = roc_auc_score(y_validation, y_probability)

        results.append({
            "Model": model_name,
            "Fold": fold_number,
            "Accuracy": accuracy,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1-score": f1,
            "AUC": auc
        })

        print(
            f"Fold {fold_number:02d}/{total_folds} completed | "
            f"Accuracy: {accuracy:.4f} | "
            f"AUC: {auc:.4f}"
        )

# ------------------------------------------------------------
# 4. STORE FOLD-LEVEL RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\nAll model evaluations completed.")

# ------------------------------------------------------------
# 5. CALCULATE MEAN AND STANDARD DEVIATION
# ------------------------------------------------------------

metrics = [
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1-score",
    "AUC"
]

summary_rows = []

for model_name, group in results_df.groupby("Model"):

    row = {"Model": model_name}

    for metric in metrics:
        row[metric + " Mean"] = group[metric].mean()
        row[metric + " SD"] = group[metric].std(ddof=1)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Format mean ± SD for easy reporting
formatted_summary = pd.DataFrame()
formatted_summary["Model"] = summary_df["Model"]

for metric in metrics:
    formatted_summary[metric] = (
        summary_df[metric + " Mean"].map(lambda v: f"{v:.4f}")
        + " ± "
        + summary_df[metric + " SD"].map(lambda v: f"{v:.4f}")
    )

print("\n" + "=" * 90)
print("MODEL PERFORMANCE: MEAN ± STANDARD DEVIATION")
print("=" * 90)

display(formatted_summary)

# ------------------------------------------------------------
# 6. PRIMARY STATISTICAL COMPARISON:
#    VOTING ENSEMBLE vs CATBOOST USING AUC
#
# Corrected resampled t-test (Nadeau-Bengio correction).
# This accounts approximately for dependence between
# repeated cross-validation fold results.
# ------------------------------------------------------------

voting_auc = (
    results_df[results_df["Model"] == "Voting Ensemble"]
    .sort_values("Fold")["AUC"].to_numpy()
)

catboost_auc = (
    results_df[results_df["Model"] == "CatBoost"]
    .sort_values("Fold")["AUC"].to_numpy()
)

differences = voting_auc - catboost_auc

n = len(differences)
mean_difference = np.mean(differences)
variance_difference = np.var(differences, ddof=1)

# For 5-fold CV, validation/train size ratio = 1/4
test_train_ratio = 1 / 4

corrected_standard_error = np.sqrt(
    (1 / n + test_train_ratio) * variance_difference
)

if corrected_standard_error == 0:
    t_statistic = 0.0 if mean_difference == 0 else np.inf
    p_value = 1.0 if mean_difference == 0 else 0.0
    ci_lower = mean_difference
    ci_upper = mean_difference
else:
    t_statistic = mean_difference / corrected_standard_error
    degrees_of_freedom = n - 1

    p_value = 2 * student_t.sf(
        abs(t_statistic),
        df=degrees_of_freedom
    )

    critical_value = student_t.ppf(
        0.975,
        df=degrees_of_freedom
    )

    ci_lower = (
        mean_difference
        - critical_value * corrected_standard_error
    )

    ci_upper = (
        mean_difference
        + critical_value * corrected_standard_error
    )

print("\n" + "=" * 70)
print("CORRECTED PAIRED COMPARISON: VOTING ENSEMBLE vs CATBOOST")
print("=" * 70)

print(f"Mean AUC difference (Voting - CatBoost): {mean_difference:.4f}")
print(f"Corrected t-statistic: {t_statistic:.4f}")
print(f"Degrees of freedom: {n - 1}")
print(f"Two-sided p-value: {p_value:.6f}")
print(f"95% confidence interval: [{ci_lower:.4f}, {ci_upper:.4f}]")

if p_value < 0.05:
    print("Interpretation: The AUC difference is statistically significant at 0.05.")
else:
    print("Interpretation: The AUC difference is not statistically significant at 0.05.")

# ------------------------------------------------------------
# 7. SAVE RESULTS TO YOUR GOOGLE DRIVE
# ------------------------------------------------------------

results_df.to_csv(
    os.path.join(folder, "Comment3_Fold_Level_Results.csv"),
    index=False
)

summary_df.to_csv(
    os.path.join(folder, "Comment3_Mean_SD_Summary.csv"),
    index=False
)

formatted_summary.to_csv(
    os.path.join(folder, "Comment3_Formatted_Mean_SD.csv"),
    index=False
)

print("\nResults saved in:")
print(folder)

print("\nTotal experiment time: "
      f"{(time.time() - start_time) / 60:.2f} minutes.")


MODEL: Logistic Regression
Fold 01/10 completed | Accuracy: 0.6338 | AUC: 0.6907
Fold 02/10 completed | Accuracy: 0.6667 | AUC: 0.7198
Fold 03/10 completed | Accuracy: 0.6768 | AUC: 0.7391
Fold 04/10 completed | Accuracy: 0.6944 | AUC: 0.7667
Fold 05/10 completed | Accuracy: 0.7190 | AUC: 0.7607
Fold 06/10 completed | Accuracy: 0.6995 | AUC: 0.7456
Fold 07/10 completed | Accuracy: 0.6540 | AUC: 0.7096
Fold 08/10 completed | Accuracy: 0.7071 | AUC: 0.7922
Fold 09/10 completed | Accuracy: 0.6515 | AUC: 0.7153
Fold 10/10 completed | Accuracy: 0.6734 | AUC: 0.7331

MODEL: Random Forest
Fold 01/10 completed | Accuracy: 0.6717 | AUC: 0.7065
Fold 02/10 completed | Accuracy: 0.7020 | AUC: 0.7399
Fold 03/10 completed | Accuracy: 0.6970 | AUC: 0.7642
Fold 04/10 completed | Accuracy: 0.6919 | AUC: 0.7688
Fold 05/10 completed | Accuracy: 0.6886 | AUC: 0.7337
Fold 06/10 completed | Accuracy: 0.6843 | AUC: 0.7551
Fold 07/10 completed | Accuracy: 0.6995 | AUC: 0.7425
Fold 08/10 completed | Accuracy:

,Model,Accuracy,Sensitivity,Specificity,Precision,F1-score,AUC
0,CatBoost,0.6981 ± 0.0211,0.7531 ± 0.0263,0.6231 ± 0.0365,0.7319 ± 0.0198,0.7421 ± 0.0185,0.7668 ± 0.0243
1,LightGBM,0.6885 ± 0.0202,0.7391 ± 0.0326,0.6195 ± 0.0301,0.7262 ± 0.0161,0.7322 ± 0.0202,0.7533 ± 0.0203
2,Logistic Regression,0.6776 ± 0.0271,0.6633 ± 0.0267,0.6971 ± 0.0367,0.7495 ± 0.0273,0.7036 ± 0.0252,0.7373 ± 0.0303
3,Random Forest,0.6913 ± 0.0147,0.7539 ± 0.0179,0.6057 ± 0.0298,0.7232 ± 0.0149,0.7381 ± 0.0122,0.7466 ± 0.0238
4,Voting Ensemble,0.6963 ± 0.0216,0.7478 ± 0.0243,0.6261 ± 0.0351,0.7320 ± 0.0201,0.7397 ± 0.0188,0.7616 ± 0.0210
5,XGBoost,0.6822 ± 0.0161,0.7351 ± 0.0255,0.6099 ± 0.0207,0.7200 ± 0.0121,0.7273 ± 0.0161,0.7451 ± 0.0207



CORRECTED PAIRED COMPARISON: VOTING ENSEMBLE vs CATBOOST
Mean AUC difference (Voting - CatBoost): -0.0052
Corrected t-statistic: -0.8309
Degrees of freedom: 9
Two-sided p-value: 0.427539
95% confidence interval: [-0.0193, 0.0089]
Interpretation: The AUC difference is not statistically significant at 0.05.

Results saved in:
/content/drive/MyDrive/BC SURVIVAL METABRIC DATASETS

Total experiment time: 2.09 minutes.


In [4]:
import os
import platform
import psutil
import sklearn
import imblearn
import xgboost
import lightgbm
import catboost

print("=" * 60)
print("COMPUTATIONAL ENVIRONMENT")
print("=" * 60)

print("Operating system:", platform.platform())
print("Python version:", platform.python_version())
print("Processor:", platform.processor() or platform.machine())

print("\nCPU information:")
print("Logical CPU cores:", os.cpu_count())

print("\nRAM information:")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

print("\nGPU information:")
try:
    import torch

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
    else:
        print("GPU: Not available (CPU runtime)")
except Exception as e:
    print("GPU information unavailable:", str(e))

print("\nLibrary versions:")
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("XGBoost:", xgboost.__version__)
print("LightGBM:", lightgbm.__version__)
print("CatBoost:", catboost.__version__)

COMPUTATIONAL ENVIRONMENT
Operating system: Linux-6.6.159+-x86_64-with-glibc2.39
Python version: 3.13.18
Processor: x86_64

CPU information:
Logical CPU cores: 2

RAM information:
Total RAM: 12.67 GB

GPU information:
GPU: Not available (CPU runtime)

Library versions:
scikit-learn: 1.6.1
imbalanced-learn: 0.14.2
XGBoost: 3.4.1
LightGBM: 4.6.0
CatBoost: 1.2.10
